In [0]:

from pyspark.sql import functions as F

silver_table = "myadlasdevcalaog.silver.customers"
gold_table = "myadlasdevcalaog.gold.customer_city_summary"

# Read Silver and Gold tables
silver_df = spark.table(silver_table)
gold_df = spark.table(gold_table)

# Check 1: Compare customer counts
silver_count = silver_df.select("customer_id").distinct().count()

gold_count = gold_df.agg(
    F.sum("total_customers").alias("total")
).first()["total"]

gold_count = gold_count or 0

print(f"Silver distinct customers: {silver_count}")
print(f"Gold summarized customers: {gold_count}")

if silver_count != gold_count:
    raise ValueError("FAILED: Silver and Gold customer counts do not match.")

# Check 2: Check duplicate customer IDs in Silver
silver_duplicates = (
    silver_df.groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicate customer IDs in Silver: {silver_duplicates}")

if silver_duplicates > 0:
    raise ValueError("FAILED: Duplicate customer IDs found in Silver.")

# Check 3: Check duplicate city rows in Gold
gold_duplicate_cities = (
    gold_df.groupBy("city")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicate city rows in Gold: {gold_duplicate_cities}")

if gold_duplicate_cities > 0:
    raise ValueError("FAILED: Duplicate city summaries found in Gold.")

print("SUCCESS: All data quality checks passed.")


Silver distinct customers: 6
Gold summarized customers: 6
Duplicate customer IDs in Silver: 0
Duplicate city rows in Gold: 0
SUCCESS: All data quality checks passed.


In [0]:

from pyspark.sql import functions as F

silver_table = "myadlasdevcalaog.silver.customers"
df = spark.table(silver_table)

# Check for duplicate customer IDs
duplicate_count = (
    df.groupBy("customer_id")
      .count()
      .filter(F.col("count") > 1)
      .count()
)

# Check for missing customer IDs or names
invalid_count = df.filter(
    F.col("customer_id").isNull()
    | F.col("customer_name").isNull()
    | (F.trim(F.col("customer_name")) == "")
).count()

print("Duplicate customer IDs:", duplicate_count)
print("Invalid customer records:", invalid_count)

if duplicate_count == 0 and invalid_count == 0:
    print("DATA QUALITY CHECK: PASSED")
else:
    raise Exception("DATA QUALITY CHECK: FAILED")


Duplicate customer IDs: 0
Invalid customer records: 0
DATA QUALITY CHECK: PASSED
